---
title: "动态规划 (DP)——二维网格"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [37]:
#| code-fold: true
from typing import List
import copy

## 📝 题目 62：不同路径

::: {.callout-caution}
### 📖 题目描述
一个机器人位于一个 `m x n` 网格的左上角 （起始点在下图中标记为 “Start” ）。

机器人每次只能向下或者向右移动一步。机器人试图达到网格的右下角（在下图中标记为 “Finish” ）。

问总共有多少条不同的路径？

**输入输出示例**：

* **示例 1**：
    * **输入**：`m = 3, n = 7`
    * **输出**：`28`

* **示例 2**：
    * **输入**：`m = 3, n = 2`
    * **输出**：`3`
    * **解释**：
      从左上角开始，总共有 3 条路径可以到达右下角。
      1. 向右 -> 向下 -> 向下
      2. 向下 -> 向下 -> 向右
      3. 向下 -> 向右 -> 向下
:::

In [14]:
class Solution62:
    def uniquePaths(self, m: int, n: int) -> int:
        dp = [[1] * n for _ in range(m)]
        for i in range(1, m):  # 从 1 开始遍历，覆盖掉内部的 1
            for j in range(1, n):
                dp[i][j] = dp[i-1][j] + dp[i][j-1]
        return dp[-1][-1]

In [15]:
#| code-fold: true
def test_62(func):
    test_cases = [
        {"desc": "最小网格", "m": 1, "n": 1, "expected": 1},
        {"desc": "单行网格", "m": 1, "n": 5, "expected": 1},
        {"desc": "单列网格", "m": 5, "n": 1, "expected": 1},
        {"desc": "示例 1", "m": 3, "n": 7, "expected": 28},
        {"desc": "示例 2", "m": 3, "n": 2, "expected": 3},
        {"desc": "正方形网格", "m": 3, "n": 3, "expected": 6},
        {"desc": "中等网格", "m": 7, "n": 3, "expected": 28},
        {"desc": "较大网格", "m": 10, "n": 10, "expected": 48620},
        {"desc": "长宽悬殊网格", "m": 2, "n": 20, "expected": 20},
        {"desc": "极限测试", "m": 23, "n": 12, "expected": 193536720}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<10} | {'实际':<10} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["m"], tc["n"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<10} | {actual:<10} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_62(Solution62().uniquePaths)

ID   | 结果     | 预期         | 实际         | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 1          | 1          | 最小网格
2    | ✅ PASS | 1          | 1          | 单行网格
3    | ✅ PASS | 1          | 1          | 单列网格
4    | ✅ PASS | 28         | 28         | 示例 1
5    | ✅ PASS | 3          | 3          | 示例 2
6    | ✅ PASS | 6          | 6          | 正方形网格
7    | ✅ PASS | 28         | 28         | 中等网格
8    | ✅ PASS | 48620      | 48620      | 较大网格
9    | ✅ PASS | 20         | 20         | 长宽悬殊网格
10   | ✅ PASS | 193536720  | 193536720  | 极限测试
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

dp[i][j] = dp[i-1][j] + dp[i][j-1]]

二维 DP 最迷人的地方就在于寻找“状态是怎么从相邻格子流转过来的”。

1. **确定 dp 数组以及下标的含义**：
   - 定义二维数组 `dp`。
   - `dp[i][j]` 表示：从起点 `(0, 0)` 出发，到达坐标 `(i, j)` （即第 `i` 行第 `j` 列）一共有多少条不同的路径。

2. **确定递推公式（状态转移方程）**：
   - 题目规定：机器人每次只能“向下”或“向右”移动。
   - 也就是说，要想走到坐标 `(i, j)`，**只有两条路能过来**：
     - 从它**上面**的格子走下来，即坐标 `(i-1, j)`。
     - 从它**左边**的格子走过来，即坐标 `(i, j-1)`。
   - 到达 `(i, j)` 的总路径数，等于到达上面格子的路径数，加上到达左边格子的路径数。
   - **状态转移方程：`dp[i][j] = dp[i-1][j] + dp[i][j-1]`**。

3. **dp 数组如何初始化**：
   - 想想看，第一行的格子（`i=0`）怎么走？因为只能向右走，所以到达第一行任何一个格子的路径只有 1 条（一直向右）。所以 `dp[0][j] = 1`。
   - 同理，第一列的格子（`j=0`）怎么走？只能一直向下走，路径也只有 1 条。所以 `dp[i][0] = 1`。

4. **确定遍历顺序**：
   - 因为 `dp[i][j]` 依赖于它上面和左边的值，所以我们需要**从左到右、从上到下**一层一层遍历。
   - 外层循环遍历行 `i`（从 1 到 m-1），内层循环遍历列 `j`（从 1 到 n-1）。

5. **空间优化（进阶）**：
   - 观察公式，计算当前行时，实际上只用到了“上一行”和“当前行前面的元素”。
   - 我们可以把二维数组压缩成一维数组，通过 `dp[j] = dp[j] + dp[j-1]` 不断覆盖更新，从而将空间复杂度降到 $O(n)$。初学阶段建议先踏踏实实写出二维版本。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(m \times n)$。我们需要使用双重循环遍历整个 `m x n` 的网格，每个格子进行一次加法运算。
* **空间复杂度**：
  * 使用二维 dp 数组：$O(m \times n)$。
  * 使用一维数组进行空间优化：$O(n)$（或者 $O(m)$，取决于你以哪个维度作为一维数组的长度）。
:::

## 📝 题目 63：不同路径 II

::: {.callout-caution}
### 📖 题目描述
给定一个 `m x n` 的整数数组 `obstacleGrid` 。

机器人初始位于 **左上角** （即 `obstacleGrid[0][0]` ）。机器人尝试探索到网格的 **右下角** （即 `obstacleGrid[m - 1][n - 1]` ）。机器人每次只能向下或者向右移动一步。

网格中的障碍物和空位置分别用 `1` 和 `0` 来表示。机器人的移动路径中不能包含任何带有障碍物的方格。

返回机器人能够到达右下角的不同路径数量。

**输入输出示例**：

* **示例 1**：
    * **输入**：`obstacleGrid = [[0,0,0],[0,1,0],[0,0,0]]`
    * **输出**：`2`
    * **解释**：
      3x3 网格的正中间有一个障碍物。
      从左上角到右下角一共有 2 条不同的路径：
      1. 向右 -> 向右 -> 向下 -> 向下
      2. 向下 -> 向下 -> 向右 -> 向右

* **示例 2**：
    * **输入**：`obstacleGrid = [[0,1],[0,0]]`
    * **输出**：`1`
:::

In [34]:
class Solution63:
    def uniquePathsWithObstacles(self, obstacleGrid: List[List[int]]) -> int:
        rows, cols = len(obstacleGrid), len(obstacleGrid[0])
        dp = [[0] * cols for _ in range(rows)]
        for i in range(rows):  # 第一行，只能从左边走过来
            if obstacleGrid[i][0] == 1:
                break
            dp[i][0] = 1
        for j in range(cols):  # 第一列，只能从上面走下来
            if obstacleGrid[0][j] == 1:
                break
            dp[0][j] = 1
        for i in range(1, rows):
            for j in range(1, cols):
                if obstacleGrid[i][j] == 0:  # 常规格子，等于上方 + 左方
                    dp[i][j] = dp[i-1][j] + dp[i][j-1]
        return dp[-1][-1]

In [35]:
#| code-fold: true
def test_63(func):
    test_cases = [
        {"desc": "起点被堵死", "grid": [[1,0],[0,0]], "expected": 0},
        {"desc": "终点被堵死", "grid": [[0,0],[0,1]], "expected": 0},
        {"desc": "单行中间有石头", "grid": [[0,1,0,0]], "expected": 0},
        {"desc": "单列中间有石头", "grid": [[0],[1],[0]], "expected": 0},
        {"desc": "示例 1 (常规)", "grid": [[0,0,0],[0,1,0],[0,0,0]], "expected": 2},
        {"desc": "示例 2 (常规)", "grid": [[0,1],[0,0]], "expected": 1},
        {"desc": "完全无障碍", "grid": [[0,0,0],[0,0,0]], "expected": 3},
        {"desc": "对角线墙壁", "grid": [[0,0,1],[0,1,0],[1,0,0]], "expected": 0},
        {"desc": "U型绕路", "grid": [[0,0,0],[1,1,0],[0,0,0]], "expected": 1}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["grid"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_63(Solution63().uniquePathsWithObstacles)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 0    | 0    | 起点被堵死
2    | ✅ PASS | 0    | 0    | 终点被堵死
3    | ✅ PASS | 0    | 0    | 单行中间有石头
4    | ✅ PASS | 0    | 0    | 单列中间有石头
5    | ✅ PASS | 2    | 2    | 示例 1 (常规)
6    | ✅ PASS | 1    | 1    | 示例 2 (常规)
7    | ✅ PASS | 3    | 3    | 完全无障碍
8    | ✅ PASS | 0    | 0    | 对角线墙壁
9    | ✅ PASS | 1    | 1    | U型绕路
-----------------------------------------------------------------
测试总结: 通过 9/9


::: {.callout-important}
### 💡 思路讲解



有障碍物意味着什么？意味着**这个格子走不通，到达这个格子的路径数为 0**。只要牢记这一点，状态转移方程和初始化稍微改动一下即可。

1. **确定 dp 数组以及下标的含义**：
   - 依然是定义二维数组 `dp`。`dp[i][j]` 表示到达坐标 `(i, j)` 的路径总数。

2. **确定递推公式（状态转移方程）**：
   - 遍历到 `(i, j)` 时，先看它是不是障碍物。
   - **如果 `obstacleGrid[i][j] == 1`（是障碍物）**：此路不通，直接让 `dp[i][j] = 0`。
   - **如果 `obstacleGrid[i][j] == 0`（不是障碍物）**：和 62 题一样，等于上面和左边的路径数之和。
   - 方程为：`dp[i][j] = dp[i-1][j] + dp[i][j-1]`。

3. **dp 数组如何初始化（这是本题最容易踩坑的地方！）**：
   - **起点/终点被堵死**：如果起点 `obstacleGrid[0][0] == 1`，连门都出不去，直接返回 0。
   - **第一行的初始化**：不仅要考虑当前格子是不是障碍物，还要考虑**前面的格子**有没有障碍物。因为只能向右走，一旦第一行某个位置出现障碍物，**它以及它右边所有的格子都永远到达不了了**。
     - 所以：`dp[0][j] = 1` 的前提是，从 `(0, 0)` 到 `(0, j)` 都没有障碍物。一旦遇到障碍物，后面的 `dp[0][j]` 都是 0。
   - **第一列的初始化**：同理，只能向下走。一旦第一列某个位置遇到障碍物，它下面的所有格子也都到达不了，全为 0。

4. **确定遍历顺序**：
   - 和 62 题一样，从左到右，从上到下。从 `i=1, j=1` 开始遍历。

5. **空间优化（进阶）**：
   - 依然可以优化到一维数组。但在内层循环计算 `dp[j] = dp[j] + dp[j-1]` 时，如果当前格子是障碍物，必须显式地把 `dp[j]` 设为 0（代表断开了上一行的传递）。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(m \times n)$。需要遍历一次整个网格。
* **空间复杂度**：
  * 二维 dp 数组：$O(m \times n)$。
  * 一维数组优化：$O(n)$。
:::

## 📝 题目 64：最小路径和

::: {.callout-caution}
### 📖 题目描述
给定一个包含非负整数的 `m x n` 网格 `grid` ，请找出一条从左上角到右下角的路径，使得路径上的数字总和为最小。

**说明**：每次只能向下或者向右移动一步。

**输入输出示例**：

* **示例 1**：
    * **输入**：`grid = [[1,3,1],[1,5,1],[4,2,1]]`
    * **输出**：`7`
    * **解释**：因为路径 1 → 3 → 1 → 1 → 1 的总和最小。

* **示例 2**：
    * **输入**：`grid = [[1,2,3],[4,5,6]]`
    * **输出**：`12`
:::

In [38]:
class Solution64:
    def minPathSum(self, grid: List[List[int]]) -> int:
        rows, cols = len(grid), len(grid[0])
        for i in range(1, rows):
            grid[i][0] += grid[i-1][0]
        for j in range(1, cols):
            grid[0][j] += grid[0][j-1]
        for i in range(1, rows):
            for j in range(1, cols):
                grid[i][j] += min(grid[i-1][j], grid[i][j-1])
        return grid[-1][-1]

In [39]:
#| code-fold: true
def test_64(func):
    test_cases = [
        {"desc": "示例 1", "grid": [[1,3,1],[1,5,1],[4,2,1]], "expected": 7},
        {"desc": "示例 2", "grid": [[1,2,3],[4,5,6]], "expected": 12},
        {"desc": "只有一格", "grid": [[5]], "expected": 5},
        {"desc": "单行网格", "grid": [[1,2,3,4]], "expected": 10},
        {"desc": "单列网格", "grid": [[1],[2],[3],[4]], "expected": 10},
        {"desc": "包含数字 0", "grid": [[0,1],[1,0]], "expected": 1},
        {"desc": "全相同数字", "grid": [[2,2],[2,2]], "expected": 6},
        {"desc": "之字形最优路径", "grid": [[1,10,10],[1,1,10],[10,1,1]], "expected": 5}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        grid_copy = copy.deepcopy(tc["grid"])
        actual = func(grid_copy)
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_64(Solution64().minPathSum)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 7    | 7    | 示例 1
2    | ✅ PASS | 12   | 12   | 示例 2
3    | ✅ PASS | 5    | 5    | 只有一格
4    | ✅ PASS | 10   | 10   | 单行网格
5    | ✅ PASS | 10   | 10   | 单列网格
6    | ✅ PASS | 1    | 1    | 包含数字 0
7    | ✅ PASS | 6    | 6    | 全相同数字
8    | ✅ PASS | 5    | 5    | 之字形最优路径
-----------------------------------------------------------------
测试总结: 通过 8/8


::: {.callout-important}
### 💡 思路讲解

和前面两题极其相似，只是从“算加法”变成了“选最小”。到达当前格子的路径总和，取决于它上面格子和左边格子中，哪一个的累积代价更小。

1. **确定 dp 数组以及下标的含义**：
   - 定义二维数组 `dp`。
   - `dp[i][j]` 表示：从左上角 `(0, 0)` 出发，到达坐标 `(i, j)` 时，路径上的**最小数字总和**。

2. **确定递推公式（状态转移方程）**：
   - 到达 `(i, j)` 只有两个方向：从上方 `(i-1, j)` 或者从左方 `(i, j-1)`。
   - 为了让总和最小，我们当然要在这两条路里挑一个“历史代价”更小的，再加上当前格子本身的代价 `grid[i][j]`。
   - **状态转移方程：`dp[i][j] = min(dp[i-1][j], dp[i][j-1]) + grid[i][j]`**。

3. **dp 数组如何初始化**：
   - **起点**：`dp[0][0] = grid[0][0]`。
   - **第一行**：只能从左边走过来，所以当前格子的最小总和，就是左边格子的最小总和加上当前格子的值：`dp[0][j] = dp[0][j-1] + grid[0][j]`。
   - **第一列**：只能从上面走下来，同理：`dp[i][0] = dp[i-1][0] + grid[i][0]`。

4. **确定遍历顺序**：
   - 依然是从左到右、从上到下一层层遍历（外层 `i` 从 1 到 m-1，内层 `j` 从 1 到 n-1）。

5. **空间优化（进阶技巧）**：
   - **方法一（一维数组）**：和 62 题一样，可以用一个一维数组滚动更新。
   - **方法二（原地修改，极限 $O(1)$）**：如果面试官允许修改原数组，我们甚至不需要创建 `dp` 数组！直接把算出来的最小总和覆盖到原来的 `grid[i][j]` 上，空间复杂度直接降为 $O(1)$！
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(m \times n)$。遍历整个网格一次。
* **空间复杂度**：
  * 使用新建的二维 dp 数组：$O(m \times n)$。
  * 原地修改原数组：$O(1)$（最优解）。
:::